# 📚 Week 1: 自动微分(autograd)与nn.Module

本notebook涵盖：
- autograd自动微分原理
- 自定义nn.Module
- 前向传播与反向传播

**目标**: 理解PyTorch如何计算梯度

In [ ]:
import torch
import torch.nn as nn
print(f"PyTorch version: {torch.__version__}")

## 1. autograd基础

In [ ]:
# y = x^2 + 2x + 1
# dy/dx = 2x + 2

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"x = {x}")

y = x ** 2 + 2 * x + 1
print(f"y = {y}")

z = y.sum()
z.backward()

print(f"\ndy/dx = {x.grad}")
print(f"\n验证: 2x + 2 = {[2*xi + 2 for xi in x.tolist()]}")

## 2. 复杂梯度计算

In [ ]:
# f(x, y) = x*y + exp(x)
# ∂f/∂x = y + exp(x)
# ∂f/∂y = x

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x * y + torch.exp(x)
f.backward()

print(f"∂f/∂x = {x.grad.item()}")  # 应该是 3 + exp(2) ≈ 10.389
print(f"∂f/∂y = {y.grad.item()}")  # 应该是 2

## 3. 梯度清零与多次反向传播

In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)

# 第一次前向
y = x ** 2
z = y.sum()
z.backward()
print(f"After first backward: grad = {x.grad}")

# 手动清零（或下次backward会累加）
x.grad.zero_()

# 第二次前向
y = x ** 3
z = y.sum()
z.backward()
print(f"After second backward: grad = {x.grad}")

## 4. 自定义nn.Module

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()  # 必须调用父类初始化
        self.layer1 = nn.Linear(10, 32)  # 10 -> 32
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(32, 1)   # 32 -> 1
        # 可选：自定义参数
        self.threshold = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = SimpleNet()
print(model)
print("\n参数:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

## 5. 前向传播测试

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNet().to(device)

# 测试前向传播
batch_size = 4
x = torch.randn(batch_size, 10).to(device)

with torch.no_grad():
    output = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output:\n{output}")

## 6. 更复杂的网络示例

In [ ]:
class ComplexNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 使用nn.Sequential简化
        self.features = nn.Sequential(
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.features(x)

model = ComplexNet()

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 🎯 练习任务

1. 创建三层全连接网络（输入784 -> 256 -> 64 -> 10）
2. 用随机数据(batch_size=16)测试前向传播
3. 打印所有参数形状
4. 计算并验证梯度

In [ ]:
# ===== 练习答案 =====

class ThreeLayerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = ThreeLayerNet()

# 参数形状
print("Parameters:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

# 测试前向传播
x = torch.randn(16, 784)
output = model(x)
print(f"\nOutput shape: {output.shape}")

# 梯度验证
loss = output.sum()  # 简单loss
loss.backward()
print("\nGradients (first layer):")
print(f"  fc1.weight.grad shape: {model.fc1.weight.grad.shape}")